# ???????? ??? (? ??????????? train/test)

???? ? ?????? ??????? ????????: ???????? ??????, ??????????, train/test split, ????????.


In [31]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [32]:
ROOT = Path().resolve()
COMPOSES_SRC = ROOT / "dissect-master" / "dissect-master" / "src"
sys.path.insert(0, str(COMPOSES_SRC))

from composes.composition.lexical_function import LexicalFunction
from composes.semantic_space.space import Space
from composes.matrix.dense_matrix import DenseMatrix

In [33]:
DATA_PAIRS = ROOT / "data" / "csv" / "serbian_verb_final_02_03_with_ruscorpora_100.csv"
VEC_PATH = ROOT / "models" / "embeddings" / "wiki.sr.vec"

df = pd.read_csv(DATA_PAIRS)

df = df[["base_verb", "prefixed_verb", "prefix"]].dropna()

df["base_verb"] = df["base_verb"].astype(str)
df["prefixed_verb"] = df["prefixed_verb"].astype(str)
df["prefix"] = df["prefix"].astype(str)

print("pairs in dataset:", len(df))

pairs in dataset: 739


In [34]:
def load_vec_subset(vec_path, wanted_words):

    wanted_words = set(wanted_words)
    found = {}

    with open(vec_path, "r", encoding="utf-8") as f:

        header = f.readline().split()
        dim = int(header[1])

        for line in f:

            word, vec = line.split(" ", 1)

            if word not in wanted_words:
                continue

            vec = np.fromstring(vec, sep=" ", dtype=np.float32)

            if vec.size != dim:
                continue

            found[word] = vec

            if len(found) == len(wanted_words):
                break

    return found, dim

In [ ]:
# ограничим размер для эксперимента

N_PAIRS = 5000
df_small = df.head(N_PAIRS).copy()

# какие слова нужны

wanted = set(df_small["base_verb"]).union(set(df_small["prefixed_verb"]))

vectors, dim = load_vec_subset(VEC_PATH, wanted)

print("vector dimension:", dim)
print("vectors loaded:", len(vectors))

vector dimension: 300
vectors loaded: 385


In [36]:
# удаляем пары без векторов


df_small = df_small[
    df_small["base_verb"].isin(vectors) &
    df_small["prefixed_verb"].isin(vectors)
].copy()

print("pairs after filtering:", len(df_small))

pairs after filtering: 169


In [ ]:
# train / test split

train_df, test_df = train_test_split(
    df_small,
    test_size=0.2,
    random_state=42
)

print("train pairs:", len(train_df))
print("test pairs:", len(test_df))

In [37]:
# создаем пространства слов

base_words = sorted(df_small["base_verb"].unique())
pref_words = sorted(df_small["prefixed_verb"].unique())

id2col = [f"d{i}" for i in range(dim)]

arg_mat = np.vstack([vectors[w] for w in base_words]).astype(np.float32)
phr_mat = np.vstack([vectors[w] for w in pref_words]).astype(np.float32)

arg_space = Space(DenseMatrix(arg_mat), base_words, id2col)
phrase_space = Space(DenseMatrix(phr_mat), pref_words, id2col)

print("arg_space:", arg_space.cooccurrence_matrix.shape)
print("phrase_space:", phrase_space.cooccurrence_matrix.shape)

arg_space: (85, 300)
phrase_space: (169, 300)


In [38]:
# подготовка train данных

train_data = list(
    train_df[["prefix", "base_verb", "prefixed_verb"]]
    .itertuples(index=False, name=None)
)

print("train examples:", len(train_data))

train examples: 135


In [39]:
# обучение модели

lf = LexicalFunction()

lf.train(
    train_data,
    arg_space=arg_space,
    phrase_space=phrase_space
)

print("learned prefixes:", len(lf.function_space.id2row))

Training lexical function...do with 10 samples
Training lexical function...iz with 21 samples
Training lexical function...na with 19 samples
Training lexical function...nad with 1 samples
Training lexical function...ob with 4 samples
Training lexical function...od with 9 samples
Training lexical function...pod with 1 samples
Training lexical function...pre with 10 samples
Training lexical function...pri with 8 samples
Training lexical function...pro with 8 samples
Training lexical function...raz with 4 samples
Training lexical function...sa with 5 samples
Training lexical function...u with 18 samples
Training lexical function...uz with 1 samples
Training lexical function...za with 16 samples
learned prefixes: 15


# ?????????????? ???? (????? ???????, ??????, ??????)

???? ? ?????????????? ????? ??? ???????????? ? ?????? ????????.


In [42]:
# подготовка кандидатов

cand_words = [
    w for w in sorted(test_df["prefixed_verb"].unique())
    if w in vectors
]

cand_mat = np.vstack([vectors[w] for w in cand_words]).astype(np.float32)

cand_norm = cand_mat / (
    np.linalg.norm(cand_mat, axis=1, keepdims=True) + 1e-9
)

In [43]:
# вспомогательные функции

def as_1d(matrix_obj):

    return np.asarray(matrix_obj.mat, dtype=np.float32).reshape(-1)


def nearest_prefixed(pred_vec, topn=10):

    v = pred_vec.astype(np.float32)
    v = v / (np.linalg.norm(v) + 1e-9)

    sims = cand_norm @ v

    best_idx = np.argsort(-sims)[:topn]

    return [(cand_words[i], float(sims[i])) for i in best_idx]


def predict_prefixed(prefix, base_verb, topn=10):

    composed_id = f"{prefix}_{base_verb}"

    composed_space = lf.compose(
        [(prefix, base_verb, composed_id)],
        arg_space=arg_space
    )

    pred_vec = as_1d(composed_space.get_row(composed_id))

    return nearest_prefixed(pred_vec, topn=topn)

In [44]:
# пример предсказания

example = test_df.sample(1, random_state=42).iloc[0]

print("\nquery:", example["prefix"], "+", example["base_verb"])
print("true:", example["prefixed_verb"])

print("\npredictions:")

print(
    predict_prefixed(
        example["prefix"],
        example["base_verb"],
        topn=10
    )
)


query: pre + gledati
true: pregledati

predictions:
[('protumačiti', 0.8610618114471436), ('prepustiti', 0.8599526882171631), ('uvažiti', 0.8579092025756836), ('proslediti', 0.8484383821487427), ('sačekati', 0.8461880683898926), ('preraditi', 0.8460593223571777), ('zamisliti', 0.8365647196769714), ('dovršiti', 0.8245900869369507), ('pročitati', 0.8230335116386414), ('nacrtati', 0.8113218545913696)]


In [45]:
# evaluation

def eval_topk(df_eval, k=10):

    hits = 0
    total = 0

    for row in df_eval.itertuples(index=False):

        prefix = row.prefix
        base = row.base_verb
        true_word = row.prefixed_verb

        try:

            nn = predict_prefixed(prefix, base, topn=k)

        except KeyError:
            continue

        total += 1

        if any(w == true_word for w, _ in nn):
            hits += 1

    return hits, total, hits / total if total else 0


hits, total, acc = eval_topk(test_df.head(300), k=10)

print("\nEvaluation")
print(f"top-10 hits: {hits}/{total}")
print("accuracy:", round(acc, 3))


Evaluation
top-10 hits: 21/34
accuracy: 0.618
